# CGC + Chiller Test

This test is for testing the water supply from the chiller to all water cooled cgc devices.

## Import, Setup Logger, Create Instances

In [1]:
# Import required modules
import sys
import os
import logging
from datetime import datetime
from pathlib import Path

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.cgc.psu.psu import PSU
from devices.cgc.sw.sw import SW
from devices.cgc.sw_HR.sw_HR import SWHR
from devices.chiller.chiller import Chiller, ChillerCommands

In [2]:
# Setup external logger
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"016_Temp_Check_GC_Chiller_{timestamp}.log"

logger = logging.getLogger(f"016_Temp_Check_GC_Chiller_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

2026-03-23 11:10:00,731 - INFO - Logger initialized.


Log file: C:\Users\ESIBDlab\PycharmProjects\esibd_bs\debugging\logs\016_Temp_Check_GC_Chiller_20260323_111000.log


## Create Device Instances

In [3]:
psu1 = PSU("psu1", com=3, port=0, logger=logger)

In [4]:
psu2 = PSU("psu2", com=4, port=1, logger=logger)

In [5]:
psu3 = PSU("psu3", com=5, port=2, logger=logger)

In [6]:
psu4 = PSU("psu4", com=6, port=3, logger=logger)

In [7]:
swA = SWHR("swA", com=7, stream=0, logger=logger)

In [8]:
swB = SW("swB", com=8, port=0, logger=logger)

In [9]:
chiller = Chiller("cgc_chiller", port="COM23", baudrate=115200, logger=logger)

2026-03-23 11:10:09,809 - INFO - Using external logger for device 'cgc_chiller'


## Connect All Devices

In [10]:
psu1.connect()

2026-03-23 11:10:19,826 - INFO - Connecting to PSU device psu1 on COM3, port 0
2026-03-23 11:10:19,970 - INFO - Successfully connected to PSU device psu1


True

In [11]:
psu2.connect()

2026-03-23 11:10:20,790 - INFO - Connecting to PSU device psu2 on COM4, port 1
2026-03-23 11:10:20,937 - INFO - Successfully connected to PSU device psu2


True

In [12]:
psu3.connect()

2026-03-23 11:10:22,631 - INFO - Connecting to PSU device psu3 on COM5, port 2
2026-03-23 11:10:22,773 - INFO - Successfully connected to PSU device psu3


True

In [13]:
psu4.connect()

2026-03-23 11:10:24,068 - INFO - Connecting to PSU device psu4 on COM6, port 3
2026-03-23 11:10:24,213 - INFO - Successfully connected to PSU device psu4


True

In [14]:
swA.connect()

2026-03-23 11:10:25,551 - INFO - Connecting to SW_HR device swA on COM7, stream 0
2026-03-23 11:10:25,680 - INFO - Successfully connected to SW_HR device swA (baud rate: 230400)


True

In [15]:
swB.connect()

2026-03-23 11:10:27,317 - INFO - Connecting to SW device swB on COM8, port 0
2026-03-23 11:10:27,468 - INFO - Successfully connected to SW device swB (baud rate: 230400)


True

In [16]:
chiller.connect()

2026-03-23 11:10:31,788 - INFO - Connecting to chiller cgc_chiller on COM23


True

## Configure and Start Chiller

In [17]:
chiller.set_temperature(8.0)

In [ ]:
chiller.start_device()

## Temperature Monitoring Loop

In [ ]:
import time

interval = 1  # seconds
try:
    while True:
        for psu in [psu1, psu2, psu3, psu4]:
            status, t0, t1, t2 = psu.get_sensor_data()
            if status == psu.NO_ERR:
                logger.info(f"{psu.device_id} Sensor0={t0:.1f} Sensor1={t1:.1f} Sensor2={t2:.1f} degC")

        for sw in [swA, swB]:
            status, t0, t1, t2 = sw.get_sensor_data()
            if status == sw.NO_ERR:
                logger.info(f"{sw.device_id} Sensor0={t0:.1f} Sensor1={t1:.1f} Sensor2={t2:.1f} degC")

        temp = chiller.read_temp()
        if temp is not None:
            logger.info(f"{chiller.device_id} Temp={temp:.2f} degC")

        time.sleep(interval)
except KeyboardInterrupt:
    logger.info("Temperature monitoring stopped")

## Stop Chiller and Disconnect

In [ ]:
chiller.stop_device()

In [27]:
psu1.disconnect()
psu2.disconnect()
psu3.disconnect()
psu4.disconnect()
swA.disconnect()
swB.disconnect()
chiller.disconnect()

2026-03-23 11:12:14,623 - INFO - Disconnecting PSU device psu1
2026-03-23 11:12:14,633 - INFO - Successfully disconnected PSU device psu1
2026-03-23 11:12:14,633 - INFO - Disconnecting PSU device psu2
2026-03-23 11:12:14,649 - INFO - Successfully disconnected PSU device psu2
2026-03-23 11:12:14,650 - INFO - Disconnecting PSU device psu3
2026-03-23 11:12:14,664 - INFO - Successfully disconnected PSU device psu3
2026-03-23 11:12:14,665 - INFO - Disconnecting PSU device psu4
2026-03-23 11:12:14,679 - INFO - Successfully disconnected PSU device psu4
2026-03-23 11:12:14,679 - INFO - Disconnecting SW_HR device swA
2026-03-23 11:12:14,694 - INFO - Successfully disconnected SW_HR device swA
2026-03-23 11:12:14,694 - INFO - Disconnecting SW device swB
2026-03-23 11:12:14,710 - INFO - Successfully disconnected SW device swB
2026-03-23 11:12:14,712 - INFO - Disconnected from chiller cgc_chiller


True